In [ ]:
# Hosted D2L setup: fetch the exact helper module used to build this notebook.
from pathlib import Path
from urllib.request import urlretrieve
from importlib.metadata import PackageNotFoundError, version
import importlib.util, os, subprocess, sys

required = ['numpy', 'pandas', 'matplotlib', 'requests', 'scipy', 'pillow', 'regex', 'tensorflow', 'keras', 'protobuf', 'ml-dtypes']
imports = {'pillow': 'PIL', 'protobuf': 'google.protobuf', 'ml-dtypes': 'ml_dtypes'}
pinned = {}
fallbacks = {'tensorflow': 'tensorflow==2.21.0', 'keras': 'keras==3.14.0', 'protobuf': 'protobuf==7.34.1', 'ml-dtypes': 'ml-dtypes==0.5.4'}
device = os.environ.get("D2L_HOSTED_DEVICE", "auto").lower()
if device not in ("auto", "cpu", "gpu"):
    raise ValueError(f"Invalid D2L_HOSTED_DEVICE={device!r}")
if device == "auto":
    try:
        gpu = (Path("/dev/nvidia0").exists() or
               subprocess.run(["nvidia-smi", "-L"], capture_output=True,
                              timeout=5).returncode == 0)
    except (FileNotFoundError, subprocess.SubprocessError):
        gpu = False
else:
    gpu = device == "gpu"
if not gpu:
    os.environ.setdefault("CUDA_VISIBLE_DEVICES", "-1")
    os.environ.setdefault("JAX_PLATFORMS", "cpu")
tensorflow_version = None
if 'tensorflow' in ("tensorflow", "jax"):
    try:
        tensorflow_version = version("tensorflow")
    except PackageNotFoundError:
        pass
# Colab's CPU image currently carries a CUDA-enabled TensorFlow wheel. Its
# first ordinary tensor operation probes CUDA and emits an error-level cuInit
# diagnostic. JAX notebooks also use TensorFlow for data loading, so overlay
# the matching CPU build in both CPU variants. Keep the provider's
# ``tensorflow`` distribution metadata: other preinstalled Colab packages
# depend on that distribution name, while both wheels expose the same module.
if not gpu and 'tensorflow' in ("tensorflow", "jax"):
    try:
        tensorflow_cpu_version = version("tensorflow-cpu")
    except PackageNotFoundError:
        tensorflow_cpu_version = None
    if (tensorflow_version is not None and
            tensorflow_cpu_version != tensorflow_version):
        subprocess.check_call([
            sys.executable, "-m", "pip", "install", "-q", "--no-deps",
            f"tensorflow-cpu=={tensorflow_version}",
        ])
if "tf-keras" in fallbacks and tensorflow_version is not None:
    fallbacks["tf-keras"] = f"tf-keras=={tensorflow_version}"
missing = []
for package in required:
    if package in pinned:
        wanted, cpu_requirement, gpu_requirement, match = pinned[package]
        requirement = gpu_requirement if gpu else cpu_requirement
        try:
            installed = version(package)
        except PackageNotFoundError:
            installed = None
        actual = (installed.split("+", 1)[0]
                  if installed is not None and match == "public" else installed)
        if actual != wanted:
            missing.append(requirement)
    elif importlib.util.find_spec(imports.get(package, package)) is None:
        missing.append(fallbacks.get(package, package))
if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing])

mismatched = []
for package, (wanted, _, _, match) in pinned.items():
    try:
        installed = version(package)
    except PackageNotFoundError:
        installed = None
    actual = (installed.split("+", 1)[0]
              if installed is not None and match == "public" else installed)
    if actual != wanted:
        mismatched.append(f"{package}={installed!r} (expected {wanted})")
if mismatched:
    raise RuntimeError("Hosted runtime setup failed: " + ", ".join(mismatched))

root = Path(".d2l-hosted") / "061a82116be00c585cab41d5b2a790e2a4f96d44"
package = root / "d2l"
package.mkdir(parents=True, exist_ok=True)
base = "https://raw.githubusercontent.com/smolix/d2l-neu/061a82116be00c585cab41d5b2a790e2a4f96d44/d2l"
for name in ('__init__.py', 'tensorflow.py'):
    target = package / name
    if not target.exists():
        urlretrieve(f"{base}/{name}", target)
if str(root.resolve()) not in sys.path:
    sys.path.insert(0, str(root.resolve()))
pythonpath = os.environ.get("PYTHONPATH", "").split(os.pathsep)
if str(root.resolve()) not in pythonpath:
    os.environ["PYTHONPATH"] = os.pathsep.join(
        [str(root.resolve()), *[entry for entry in pythonpath if entry]]
    )


# GPUs, Devices, and Memory

Since that section our `Trainer` has accepted an argument
`num_gpus` and ignored it. Nobody was harmed: the models of the earlier
chapters were small enough that it hardly mattered where they ran. For larger
models it matters a great deal. A current desktop GPU multiplies two
$4096 \times 4096$ matrices in under 2 milliseconds, where a CPU takes a
fifth of a second or more (that section measures both).
Unfortunately the speed comes with a constraint: the GPU has its own memory,
24 GB on such a card, and every array it computes with must be there.

Will a given model fit? The model with 8 billion parameters that this chapter
keeps returning to needs 16 GB for its weights in bf16, which fits. Training
it with Adam in fp32 needs 128 GB by the count of that section,
before a single activation is stored. That does not fit. Whether a model fits
on a device, and whether the device computes or waits, is decided by where
each array lives and how many bytes each array takes.

We ask three questions in turn. Where does an array live, and how does it move
between the CPU and a GPU? When does the CPU wait for the GPU, and how do we
time GPU work correctly? What fills GPU memory during training, and how can we
trade computation for memory? The answers go into the device-aware `Trainer`
that every later chapter uses. that section and
that section counted the bytes per parameter and per number; this
section puts those bytes on a device. Training on several GPUs at once follows
in that section. All examples fall back to the CPU when no GPU
is present. They then show that the code runs, not how fast it runs on a GPU.

In [ ]:
# Silence TensorFlow's C++ info and warning messages. TensorFlow reads this
# variable when it is first imported, so this cell runs before the imports.
import os
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'

In [ ]:
import math
import numpy as np
import tensorflow as tf
from d2l import tensorflow as d2l

In [ ]:
import time

## Devices and Placement

### Device Objects

Every array lives on a *device*: the CPU together with main memory, or one GPU
together with its own memory. To see which GPUs a machine has, run
`nvidia-smi` in a shell; it lists every card, its memory, and the processes
using it. Each framework has its own names for devices. The names are soon
forgotten; what matters in this part of the section is a single fact, which we
reach in a moment: an operation runs on one device, and all its operands must
be there.

TensorFlow names devices with strings, `'/CPU:0'` for the CPU and `'/GPU:0'`
for the first GPU. `tf.device` turns a name into a *scope*: operations
executed inside it run on the named device and store their results there.

We wrap device construction in two functions that the rest of the book uses,
and a third counts the GPUs that are available.

In [ ]:
def cpu():
    """Get the CPU device."""
    return tf.device('/CPU:0')

def gpu(i=0):
    """Get a GPU device."""
    return tf.device(f'/GPU:{i}')

def num_gpus():
    """Get the number of available GPUs."""
    return len(tf.config.experimental.list_physical_devices('GPU'))

Constructing a device scope always succeeds, whether or not the hardware
exists. Even an operation inside the scope of a missing GPU does not fail:
TensorFlow's default *soft placement* runs it on the CPU instead. Convenient
as this is, it can hide a placement mistake, so inspect `x.device` when
placement matters.

With the count we write two functions that fall back to the CPU when the
requested GPUs do not exist. They let the same code run on a laptop and on a
GPU server.

In [ ]:
def try_gpu(i=0):
    """Return gpu(i) if exists, otherwise return cpu()."""
    if num_gpus() >= i + 1:
        return gpu(i)
    return cpu()

def try_all_gpus():
    """Return all available GPUs, or [cpu(),] if no GPU exists."""
    devices = [gpu(i) for i in range(num_gpus())]
    return devices if devices else [cpu()]

try_gpu(), try_gpu(10), try_all_gpus()

Other accelerators, such as the GPUs of Apple-silicon Macs and Google's TPUs,
have device names of their own. This book uses the CPU and CUDA GPUs, and the
placement rules below carry over to the others.

### Arrays on Devices

The property `device` tells where an array lives. Where a new array is created
by default depends on the framework.

By default, eager TensorFlow creates tensors on the first GPU if there is one
and on the CPU otherwise.

It is better to create data directly on the device where it is used than to
create it on the CPU and copy it: the array then never occupies main memory or
crosses the bus between CPU and GPU.

To create a tensor on a particular device, create it inside that device's
scope.

We put an array `X` on the first GPU and an array `Y` on the second, and add
them. On a machine with fewer GPUs, `try_gpu(1)` returns the CPU, and the
examples below run without the second device.

In [ ]:
with try_gpu():
    X = tf.ones((2, 3))
with try_gpu(1):
    Y = tf.random.uniform((2, 3))
print((X + Y).device)

The addition succeeds. Outside a device scope, TensorFlow runs an operation
on its default device, the first GPU, and its default *device policy* copies
any operand that lives elsewhere, without a warning.
`tf.config.experimental.set_device_policy('explicit')` turns such silent
copies into errors.

### Copying Between Devices

A transfer between devices is slow compared with arithmetic on either of
them (that section measures how slow), so a framework that
copied silently would hide a cost at the place where it occurs. We copy
explicitly instead, as in the figure, and then add.

![X lives on GPU 0 and Y on GPU 1. The copy Z of X on GPU 1 (dashed) lets Y + Z run entirely on GPU 1.](https://raw.githubusercontent.com/smolix/d2l-neu/notebooks/img/bg-copyto.svg)

In [ ]:
with try_gpu(1):
    Z = tf.identity(X)  # The explicit copy: tf.identity in the target's scope
    print((Y + Z).device)  # The scope, not the operands, decides the device

The sum lives on the second GPU, where both of its operands are. The rule that
follows is to move data to the device at the boundary of a computation, once
(a move to the device where the data already are costs next to nothing), and
to keep the operations inside a training loop on one device.


### Models on a Device

A model computes with its parameters and its inputs together, so its
parameters must live on the device of its data. Every framework places a
whole model at once, without our listing its parameters.

A Keras model has no method to move it: its layers create their variables at
the first call, on the device where that call runs. So we run the first
forward pass inside a device scope, and the variables stay there.

In [ ]:
net = tf.keras.Sequential([tf.keras.layers.Dense(1)])
with try_gpu():
    Y_hat = net(X)  # The first call creates the variables inside the scope
Y_hat

The input and the parameters are on the same device, so the output is
computed and stored there too.

Placement in TensorFlow follows execution. A tensor created during the
forward pass lives where the surrounding operations run, and a Keras
optimizer creates its state at the first `apply_gradients`, next to the
variables it updates. The one rule is the first-call rule above: build the
variables under the device scope you intend, because moving them afterwards
means rebuilding the model.

## Asynchronous Execution

### The Execution Queue

A GPU does not execute operations at the moment Python calls them. Python
places each operation in a queue on the device and moves on, while the GPU
works through the queue in order. This lets Python prepare the next operation
while the GPU computes, but it also means that the time a Python statement
takes says nothing about the time its computation takes. We queue 32 products
of $4096 \times 4096$ matrices and read the clock twice: when Python has
issued them, and when the device has finished them.

In [ ]:
if num_gpus() > 0:
    with gpu():
        A = tf.random.normal((4096, 4096))
        A @ A
    tf.test.experimental.sync_devices()  # Warm up and wait
    t = time.perf_counter()
    with gpu():
        for _ in range(32):
            B = A @ A
    print(f'time to queue 32 matrix products: {time.perf_counter() - t:.4f} sec')
    tf.test.experimental.sync_devices()
    print(f'time until they all finished:     {time.perf_counter() - t:.4f} sec')
else:
    print('No GPU: eager operations on the CPU run synchronously.')

GPU operations are asynchronous, and `tf.test.experimental.sync_devices()`
waits until every queued operation has finished. Eager operations on the CPU
run synchronously, which is why the cell needs a GPU.

Python issued the 32 products in a small fraction of the time the device
needed to compute them. So when does Python wait at all? Only when it needs a
value: converting a result to a Python number or a NumPy array, printing it,
or branching on it in an `if`. Each such *synchronization point* blocks Python
until the queue up to that value has drained, and then the device sits idle
until Python sends more work, as the figure shows. A
training loop that prints `loss.item()` after every step synchronizes once per
step. This is why the steps of the book's `Trainer` return their losses as
arrays: `fit_epoch` adds them up on the device in a `d2l.Averager` and
converts the sums into Python numbers once per epoch
(that section).

![Python queues operations op1 through op4 and continues while the GPU runs them in order. loss.item() is a synchronization point: the CPU blocks until the GPU has drained its queue; Python then resumes, and the GPU idles until it receives more work.](https://raw.githubusercontent.com/smolix/d2l-neu/notebooks/img/bg-async-queue.svg)

Asynchronous execution also dictates how to time GPU code, and a timing that
ignores it measures the queue, not the computation. Run the computation once
before timing it, so that one-time costs such as memory allocation, library
initialization, and compilation are excluded. Wait for the device before
starting and before stopping the clock. Time several repetitions and divide,
and read the clock with `time.perf_counter`, whose resolution is far finer
than a millisecond. The cell above warms up, waits, and reads `perf_counter`,
but it reports the total for its 32 products without dividing; the timings
later in this section follow all four rules.

`Benchmark` packages these rules for this and later chapters. It calls a
function `warmup` times, waits for the device, times `repeats` further calls,
waits again, and stores the average time per call in its attribute `time`.

In [ ]:
class Benchmark:
    """Time a callable: warmup, then device-synchronized average seconds."""
    def __init__(self, f, warmup=3, repeats=10, desc='time'):
        self.desc = desc
        for _ in range(warmup):
            f()
        tf.test.experimental.sync_devices()
        t0 = time.perf_counter()
        for _ in range(repeats):
            f()
        tf.test.experimental.sync_devices()
        self.time = (time.perf_counter() - t0) / repeats

    def __repr__(self):
        return f'{self.desc}: {1000 * self.time:.2f} ms/call'

We put `Benchmark` to work at once on two claims made so far: that a GPU
multiplies matrices far faster than a CPU, and that a transfer between devices
is slow compared with arithmetic. We time the product of the queue experiment
on the CPU and on the GPU, the same product in a 16-bit format, and the copy
of the matrix from main memory to the GPU. On the GPU, the fp32 product runs
in tf32 mode on the tensor cores (that section).

In [ ]:
if num_gpus() > 0:
    with cpu():
        A_cpu = tf.identity(A)  # The same matrix in main memory
    A16 = tf.cast(A, tf.bfloat16)
    def cpu_product():
        with cpu():
            return A_cpu @ A_cpu
    @tf.function  # Eager mode would reuse a cached copy of A_cpu on the GPU
    def copy_to_gpu():
        with gpu():
            return tf.identity(A_cpu)
    print(Benchmark(cpu_product, warmup=1, repeats=3,
                    desc='product on the CPU, fp32'))
    print(Benchmark(lambda: A @ A, desc='product on the GPU, tf32'))
    print(Benchmark(lambda: A16 @ A16, desc='product on the GPU, bf16'))
    print(Benchmark(copy_to_gpu, desc='copy from CPU to the GPU'))
else:
    print('No GPU: there is nothing to compare the CPU with.')

TensorFlow uses tf32 for fp32 matrix products by default.

The GPU computes the product in under 2 ms. The CPU takes between a fifth of a
second and two seconds, a hundred to a thousand times as long; where in that
range it lands depends on how many CPU cores take part. In 16 bits the
product takes about half the time that it takes in tf32, which is the speed
that that section promised and could not show on a CPU.

The last line is the one to remember. Copying the matrix to the GPU takes
about three to five times as long as multiplying it there, although the
product performs 4096 multiplications and additions for every number copied.
An ordinary copy from main memory first moves the data into *pinned*
(page-locked) memory, which the GPU's copy engine can read directly. Data
that are pinned already skip that step, and their copy can overlap with
computation, as that section measures.

## GPU Memory

### Three Views of Memory

How much memory does a program use on a GPU? The answer depends on who is
asked. A GPU *allocator* requests memory from the CUDA driver in large
blocks and hands out pieces of them, and when an array is freed, it keeps the
block for the next array instead of returning it, because requests to the
driver are slow. There are therefore three nested views of memory,
the figure: the bytes held by live arrays, the bytes the
allocator has reserved, and the bytes the driver has given to the process,
which `nvidia-smi` reports.

![The three memory views nest: the driver's allocation to the process, which nvidia-smi reports, contains the allocator's reserved blocks, which contain the live arrays.](https://raw.githubusercontent.com/smolix/d2l-neu/notebooks/img/bg-allocator.svg)

TensorFlow reserves GPU memory for its own allocator.
`tf.config.experimental.get_memory_info('GPU:0')` returns a dictionary whose
entry `'current'` counts the bytes of live tensors and whose entry `'peak'`
is their largest value so far. On the CPU there is no such allocator, and the
call raises an error.

We create an array of 1 GiB and delete it again:

In [ ]:
if num_gpus() > 0:
    def report(tag):
        info = tf.config.experimental.get_memory_info('GPU:0')
        print(f'{tag}: {info["current"] / 2**20:7.1f} MiB in use, '
              f'{info["peak"] / 2**20:7.1f} MiB peak')
    with gpu():
        big = tf.zeros((256, 1024, 1024))  # 1 GiB of float32
    report('after creating big')
    del big
    report('after deleting it ')
else:
    print('No GPU: get_memory_info() tracks only GPU and TPU allocators.')

The count of live bytes falls by a gibibyte when `big` is deleted, to the 160
MiB of the matrices `A`, `B`, and `A16` of the timing cells above, while
`nvidia-smi` does not move: the block returns to TensorFlow's allocator, not
to the driver, and TensorFlow keeps its reservation until the process ends. To share a GPU
with another process, call
`tf.config.experimental.set_memory_growth(gpu, True)`, which grows the
reservation on demand instead, before the first computation.

The memory that limits what fits is the outermost view, since the driver
cannot give a process more than the device has. The counts of live bytes are
what we use to find out where the memory goes. Of these, the one that decides
whether a computation fits is its *peak*, the most memory it holds at any
moment.

### Memory During Training

For a model with $N$ parameters that trains with Adam in fp32,
that section counted $16N$ bytes: $4N$ each for the weights and
their gradients, and $8N$ for Adam's two running averages. The remaining
term, the *activations*, is of a different kind. Backpropagation needs the
intermediate results of the forward pass to compute the gradients, so it
stores them until the backward pass has used them. Their size grows with the
minibatch size and with the width of every layer, while $16N$ does not. Let us
predict the terms before we measure them. We take an MLP with three hidden
layers of width 4096 on inputs of width 1024, $N \approx 3.8 \times 10^7$
parameters, so that $4N \approx 144$ MiB, at a minibatch size of 4096. The
activations that backpropagation keeps are mainly the outputs of the three
hidden layers, each a $4096 \times 4096$ array of fp32 numbers, 64 MiB apiece:
192 MiB in all, about a third more than the weights.

We read `'current'` at four points of one training step. `tf.GradientTape`
records the forward pass and holds its intermediate results until we ask it
for the gradients, so the activations appear in the second reading.

In [ ]:
if num_gpus() > 0:
    start = tf.config.experimental.get_memory_info('GPU:0')['current']
    def in_use():  # Memory in use beyond that at the start of the cell
        info = tf.config.experimental.get_memory_info('GPU:0')
        return f'{(info["current"] - start) / 2**20:7.1f} MiB'
    with gpu():
        net = tf.keras.Sequential(
            [tf.keras.layers.Dense(4096, activation='relu'),
             tf.keras.layers.Dense(4096, activation='relu'),
             tf.keras.layers.Dense(4096, activation='relu'),
             tf.keras.layers.Dense(10)])
        X = tf.random.normal((4096, 1024))
        y = tf.random.uniform((4096,), 0, 10, dtype=tf.int32)
        net(X)  # The first call creates the variables
    optimizer = tf.keras.optimizers.Adam()
    loss_fn = tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True)
    print('weights           ', in_use())
    with tf.GradientTape() as tape:
        loss = loss_fn(y, net(X, training=True))
    print('+ activations     ', in_use())
    grads = tape.gradient(loss, net.trainable_variables)
    print('+ gradients       ', in_use())
    optimizer.apply_gradients(zip(grads, net.trainable_variables))
    print('+ optimizer state ', in_use())
else:
    print('No GPU: run this cell on a GPU to see the four readings.')

The first reading is the weights, about $4N$ bytes, plus the input
minibatch. The second adds the activations that the tape holds, here about a
third more than the weights. The third *falls*: `tape.gradient` frees the
recorded forward pass and leaves gradients of the same size as the weights.
The fourth adds about $8N$ at the first `apply_gradients`, when Adam creates
its two running averages. From then on every step rises to the second
reading and falls back to the third; weights, gradients, and optimizer state
stay.

The minibatch size is the first memory control to try, because it scales the
activations without changing the model. If memory instead grows from one step
to the next, something keeps the arrays of earlier steps alive. The usual
suspect is a Python list of losses: a loss that still carries its computation
graph keeps the activations of its step alive with it, so detach the loss
before storing it.

### Activation Checkpointing

The activations have a second control. Backpropagation stores every
intermediate result only to read it once, on the way back. *Activation
checkpointing* [@Chen.Xu.Zhang.ea.2016] stores less: it keeps only the
inputs of selected segments of the network, and during the backward pass it
reruns the forward computation of each segment to regenerate the activations
that segment needs. (The name has nothing to do with the checkpoint files of
that section: here a checkpoint is an activation kept in memory.)
For a stack of $K$ equally sized layers cut into segments of $s$ layers, the
backward pass holds the $K/s$ segment inputs plus the activations of one
segment, about $K/s + s$ layer activations instead of $K$. The sum is smallest
at $s = \sqrt{K}$, where stored activations fall from $O(K)$ to $O(\sqrt{K})$.
The saving costs computation, namely a second forward pass through every
segment, the figure. How much longer does a step
take? A backward pass costs about twice a forward pass: for a linear layer it
computes two matrix products, one for the gradient of the weights and one for
the gradient of the input, where the forward pass computes one. A step then
costs about four forward passes instead of three, a third more. This is a
rather crude count, and the exact overhead depends on the layers and the
hardware.

![(a) Standard backpropagation stores the activations of every layer. (b) Checkpointing stores only the inputs of the segments (filled) and recomputes the activations inside each segment (open) during the backward pass. With $K$ layers in segments of $\sqrt{K}$ layers, stored activations fall from $O(K)$ to $O(\sqrt{K})$ at the cost of one extra forward pass.](https://raw.githubusercontent.com/smolix/d2l-neu/notebooks/img/bg-activation-checkpoint.svg)

Checkpointing suits deep stacks of identical blocks, such as the residual
stack of that section and the transformers of later
chapters. We use the residual block of that section,
$\mathbf{X} + \textrm{Linear}(\textrm{ReLU}(\textrm{Linear}(\mathbf{X})))$,
with its weights drawn from $\mathcal{N}(0, 0.01^2)$ and zero biases in every
framework, so that all tabs build the same model. A function runs a stack of
blocks in segments, with or without checkpointing.

In [ ]:
class ResidualBlock(tf.keras.layers.Layer):
    def __init__(self, num_hiddens, **kwargs):
        super().__init__(**kwargs)
        normal = tf.keras.initializers.RandomNormal  # One instance per layer
        self.body = tf.keras.Sequential([
            tf.keras.layers.Dense(num_hiddens, activation='relu',
                                  kernel_initializer=normal(stddev=0.01)),
            tf.keras.layers.Dense(num_hiddens,
                                  kernel_initializer=normal(stddev=0.01))])

    def call(self, X):
        return X + self.body(X)

def run_stack(blocks, X, use_checkpoint=False, segment_size=4):
    for start in range(0, len(blocks), segment_size):
        # Bind this segment's blocks now: the backward pass calls segment
        # again after the loop has moved on
        def segment(X, segment_blocks=blocks[start:start + segment_size]):
            for blk in segment_blocks:
                X = blk(X)
            return X
        f = tf.recompute_grad(segment) if use_checkpoint else segment
        X = f(X)
    return X

`tf.recompute_grad(segment)` returns a function that computes the same values
as `segment` but tells the tape to store only its input, and reruns `segment`
during the backward pass. It does not save the state of the random number
generator, so a segment with stateful dropout can draw a different mask when
it is recomputed.

Recomputation must not change the result. We check that the gradients
through a stack of eight blocks, run as two checkpointed segments, equal the
ordinary ones.

In [ ]:
blocks = [ResidualBlock(64) for _ in range(8)]
X = tf.random.normal((32, 64))

def stack_grads(use_checkpoint):
    with tf.GradientTape() as tape:
        loss = tf.reduce_sum(run_stack(blocks, X, use_checkpoint))
    params = [v for blk in blocks for v in blk.trainable_variables]
    return tape.gradient(loss, params)

for g, g_ckpt in zip(stack_grads(False), stack_grads(True)):
    assert np.allclose(g, g_ckpt)
print('checkpointed gradients match the ordinary ones')

The check runs on the CPU too. The memory effect needs a GPU and a stack deep
enough for a clear difference: sixteen blocks of width 1024 at a minibatch
size of 8192, in segments of four blocks. We report the *step memory*, the
peak memory of a step beyond the parameters and the input it starts from,
and the time per step that `Benchmark` measures.

In [ ]:
if num_gpus() > 0:
    with gpu():
        blocks = [ResidualBlock(1024) for _ in range(16)]
        X = tf.random.normal((8192, 1024))
        run_stack(blocks, X)  # Create the variables
    params = [v for blk in blocks for v in blk.trainable_variables]
    @tf.function  # One graph for each value of use_checkpoint
    def step(use_checkpoint):
        with tf.GradientTape() as tape:
            loss = tf.reduce_sum(run_stack(blocks, X, use_checkpoint))
        return tape.gradient(loss, params)
    for use_checkpoint in (False, True):
        step(use_checkpoint)  # Trace and warm up
        tf.test.experimental.sync_devices()
        start = tf.config.experimental.get_memory_info('GPU:0')['current']
        tf.config.experimental.reset_memory_stats('GPU:0')
        timing = Benchmark(lambda: step(use_checkpoint), warmup=0)
        peak = tf.config.experimental.get_memory_info('GPU:0')['peak'] - start
        print(f'checkpointing={use_checkpoint!s:5}  step memory '
              f'{peak / 2**20:5.0f} MiB, {timing.time * 1e3:5.1f} ms per step')
else:
    print('No GPU: the peak-memory comparison needs a GPU.')

Like all TensorFlow code that we time, the step is compiled with
`tf.function`.

Without checkpointing, the step holds the activations of all sixteen blocks.
With segments of four blocks, it holds the four segment inputs plus the
activations of one segment, and its memory falls from about 1.1 GiB to between
0.4 and 0.5 GiB. The step takes about a quarter longer, a little less than
the third that our crude count of forward passes predicted. This trade lets a
model train when its activations would otherwise exceed the memory of the
device, and it is common in the training of large transformers.

## Device-Aware Trainer

The `Trainer` can at last put its argument `num_gpus` to use. It needs nothing
beyond the rules of this section: the parameters are on the device before
training starts, and every minibatch is moved there at the start of its step.

`prepare_batch` copies every minibatch with `tf.identity` inside the device
scope: `tf.data` produces minibatches on the CPU, and without the copy every
operation on them would copy them silently. The variables need no code of
their own. Keras creates them at the first call, and the `Trainer` makes that
call, a forward pass on a minibatch from `prepare_batch`, when it compiles its
steps.

In [ ]:
@d2l.add_to_class(d2l.Trainer)
def __init__(self, max_epochs, num_gpus=0, gradient_clip_val=0):
    self.save_hyperparameters()
    self.gpus = [d2l.gpu(i) for i in range(min(num_gpus, d2l.num_gpus()))]
    self.board = d2l.ProgressBoard(xlabel='epoch', ylabel='loss')

@d2l.add_to_class(d2l.Trainer)
def prepare_batch(self, batch):
    if self.gpus:
        # tf.data.Dataset emits batches on CPU. Re-wrap them inside the
        # GPU device context so subsequent ops keep their inputs on-device
        # rather than incurring an implicit copy each step.
        with self.gpus[0]:
            batch = [tf.identity(a) for a in batch]
    return batch

The expression `min(num_gpus, d2l.num_gpus())` makes the request an upper
limit rather than a demand. On a machine without a GPU, `self.gpus` is
empty, nothing is moved to a GPU, and training runs on the CPU. Every
`Trainer(num_gpus=1)` in the rest of the book relies on this fallback.

Does a GPU pay off in training? The product of two large matrices ran more
than a hundred times faster on the GPU than on the CPU, but a training step is
more than one large product. We answer for a classifier built from the residual
blocks of that section: a hidden layer of width
256, two residual blocks, and 10 outputs, with all weights drawn from
$\mathcal{N}(0, 0.01^2)$ and zero biases, as for the MLPs of earlier chapters.

In [ ]:
class ResMLPClassifier(d2l.Classifier):
    def __init__(self, num_hiddens=256, num_blocks=2, lr=0.1):
        super().__init__()
        self.save_hyperparameters()
        normal = tf.keras.initializers.RandomNormal  # One instance per layer
        self.net = tf.keras.Sequential([
            tf.keras.layers.Flatten(),
            tf.keras.layers.Dense(num_hiddens, activation='relu',
                                  kernel_initializer=normal(stddev=0.01)),
            *[ResidualBlock(num_hiddens) for _ in range(num_blocks)],
            tf.keras.layers.Dense(10, kernel_initializer=normal(stddev=0.01))])

We train the model on Fashion-MNIST with `num_gpus=0` and with `num_gpus=1`,
and time one epoch after one epoch of warm-up.

In [ ]:
data = d2l.FashionMNIST(batch_size=256)

def epoch_time(num_gpus):
    model = ResMLPClassifier()
    trainer = d2l.Trainer(max_epochs=1, num_gpus=num_gpus)
    trainer.board.display = False
    # TensorFlow runs operations on a GPU whenever there is one, and
    # num_gpus=0 only keeps the minibatches where they are: the CPU scope
    # keeps the whole run on the CPU
    with try_gpu() if num_gpus else cpu():
        trainer.fit(model, data)  # The first epoch warms up and compiles
        return Benchmark(trainer.fit_epoch, warmup=0, repeats=1).time

for gpus in (0, 1):
    print(f'num_gpus={gpus}: {epoch_time(num_gpus=gpus):.1f} sec per epoch')

This is a sobering result. On the GPU an epoch runs at most a few times as
fast as on the CPU, far from the factor of more than a hundred that the GPU
gained on the large matrix product. A step of this model multiplies matrices
with at most $784 \times 256$ entries, too little work to keep the GPU busy,
and much of an epoch goes into loading the data and launching the many small
operations of each step. A GPU pays off when large matrix products dominate
the work, as they do in the larger models of the chapters to come.

The memory of this run follows the accounting above: weights, gradients, and
the state of the optimizer, which for SGD is nothing, plus the activations of
one minibatch. It is a small model, and it fits with room to spare. And the
8-billion-parameter model of the opening? A smaller minibatch and activation
checkpointing shrink its activations, but its 128 GB of weights, gradients,
and optimizer state are needed before the first activation is stored, and no
setting of this section touches them. Note that the `Trainer` would not have
warned us: it cannot tell whether a model fits before it runs out of memory.
That estimate we still make by hand, from the bytes per parameter and the size
of the activations. Training on more than one device is taken up in
that section.

## Summary

Every array and every parameter lives on a device, and an operation combines
arrays on one device. Copies between devices are explicit and slow compared
with arithmetic, so they belong at the boundary of a computation, not inside
the training loop. A GPU executes its queue asynchronously, so Python waits
only at synchronization points, such as converting a result to a Python
number, and correct timing needs a warm-up, explicit waits, and repetitions.
GPU memory during training holds the weights, their gradients, and the
optimizer state, which depend only on the model, plus activations that grow
with the minibatch size. Activation checkpointing stores only the inputs of
segments and recomputes the rest, which lowers the peak memory at the cost of
one extra forward pass. The device-aware `Trainer` moves the model once and
every minibatch per step, and it falls back to the CPU when no GPU is present.

TensorFlow copies between devices silently by default, so in TensorFlow
explicit copies are a discipline rather than a rule the framework enforces.

## Exercises

1. [code] **Peak memory.** Take a four-layer MLP with widths 1024, 4096,
   4096, 4096 and 10 outputs, trained with Adam, as
   in that section. Using the accounting of that
   subsection, predict the peak memory of one training step at batch sizes
   64, 256, and 1024, counting weights, gradients, optimizer state, and the
   activations the backward pass keeps. Then measure the peak on a GPU.
   Where does the prediction break down, and what did it omit?

2. **Checkpointing arithmetic.** Work out the schedule of
   the figure on paper for a stack of sixteen
   `ResidualBlock`s of width 1024 at batch size 8192.
    1. Name the arrays inside one block that the backward pass must keep,
       and count their entries.
    1. Compute the activation memory held at the peak of the backward pass
       without checkpointing and with segments of one, two, four, and eight
       blocks. State how much forward computation each schedule repeats.
    1. Which segment size would you choose here, and how does it relate to
       the rule of segments of about $\sqrt{K}$ layers in
       that section?

3. [code] **Asynchronous dispatch.** ● Use the timing rules of
   that section for both parts.
    1. For $n$ from 256 to 8192, time a round trip of an $n \times n$
       matrix from CPU to GPU and back, and time one $n \times n$ matrix
       product on the device. Find the $n$ at which the two times are
       equal, extrapolating from how each time grows with $n$ if they do
       not cross in this range. Predict this crossover from your GPU's
       published host-to-device bandwidth and matrix-product throughput,
       and explain any gap between prediction and measurement.
    1. If you have two GPUs, time 1000 matrix products of two
       $4096 \times 4096$ matrices executed on one GPU, then 500 on each of
       two GPUs issued from the same loop. Measure the scaling, explain how
       asynchronous dispatch lets Python issue work to both devices, and
       identify sources of deviation from linear scaling. Guard the
       experiment with `num_gpus() >= 2`.

4. [code] **Precision crossover.** ● Adapt the `train` function of
   that section to a GPU: place the network and every batch on
   the device, so that the 16-bit computation runs there, and let the
   caller choose the hidden width of the MLP and the batch size. Time ten
   steps of its fp32 run and of its mixed-precision run with `Benchmark`,
   after one untimed step.
    1. For hidden widths 256 and 4096, sweep the batch size from 8 to 4096
       in powers of two and plot the mixed-precision speedup over fp32
       against the batch size, one curve per width.
    1. Below which batch size, if any, is mixed precision slower than fp32
       at each width? Explain the reversal and why the crossover moves with
       the width.

    *Adapted from the PyTorch recipe
    [Automatic Mixed Precision](https://docs.pytorch.org/tutorials/recipes/recipes/amp_recipe.html).*

5. [code] **Synchronization points.** ● Consider the training run of
   `ResMLPClassifier` in that section on a GPU.
    1. List every synchronization point in one training step, reading
       `Trainer.fit_epoch`, `Trainer.prepare_batch`, and `Averager` as
       well as the model code. Classify each as unavoidable for a correct
       loop or as an avoidable instrumentation artifact, and justify each
       classification.
    1. Suppose the loop reads the loss into a Python float after every
       step. Predict from your list how much this readout slows down the
       default `ResMLPClassifier` and
       `ResMLPClassifier(num_hiddens=2048, num_blocks=8)`. Then time one
       epoch of each model with and without the readout, and explain the
       four timings.

    *Adapted from the PyTorch documentation
    [CUDA semantics](https://docs.pytorch.org/docs/stable/notes/cuda.html).*
<!-- d2l:prose id=gpus-devices-memory-md-43 fw=pytorch, tensorflow -->

6. [code] **Activation checkpointing.** Revisit the comparison
   of that section.
    1. Increase the batch size until the run without checkpointing raises
       an out-of-memory error. How much further can the checkpointed run go
       before it does too? Explain the ratio from what each variant stores.
    1. Run `run_stack` with `segment_size` equal to 1, 4, and the number of
       blocks, and once without checkpointing. Plot step memory against
       step time for the four configurations. Which segment size would you
       choose here, and why?

[Discussions](https://d2l.discourse.group/t/36758)